# tensor-unbind — ex8: RGB to grayscale with side-by-side plot

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-unbind`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-unbind`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-unbind"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch unbind — quick refresher

`x.unbind(dim=k)` returns a tuple of `x.shape[k]` view-tensors with axis `k` removed. The result is a *Python tuple*, not a tensor — perfect for destructuring named components (`origin, direction = rays.unbind(dim=1)`) or for fanning a batched tensor into per-head / per-channel slices.

**Compared to `select`.** `unbind(dim=k)[i]` ≡ `select(k, i)`. Use `select` when you want ONE slice; use `unbind` when you want ALL of them. Both return views (no copy), so writes through the view alias the source.

### Exercise 8 — RGB to grayscale with side-by-side plot

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `unbind(dim=-1)` to destructure an `(H, W, 3)` RGB image into named R/G/B channels and compute the luma-weighted grayscale conversion.
> Keywords: channels, grayscale, luma, visualization
> ```

**KCs targeted:** `unbind-explicit-dim`, `unbind-tuple-destructure`

Implement `ex8_rgb_to_grayscale(img)`. The canonical channel-destructure pattern for image processing:

1. `img` has shape `(H, W, 3)` — height × width × RGB channels, values in `[0, 1]`.
2. Use `img.unbind(dim=-1)` to get `r, g, b` as three `(H, W)` tensors.
3. Compute the ITU-R BT.601 luma:
   `gray = 0.299 * r + 0.587 * g + 0.114 * b`
4. Return the `(H, W)` grayscale tensor.

Input: `(H, W, 3)` float tensor, values in `[0, 1]`.
Output: `(H, W)` float tensor, values in `[0, 1]`.

The visualization renders the original RGB and the grayscale result side by side so you can verify the conversion looks reasonable on a synthetic image.

In [ ]:
def ex8_rgb_to_grayscale(img: Tensor) -> Tensor:
    """Convert (H, W, 3) RGB to (H, W) grayscale via BT.601 luma."""
    raise NotImplementedError()


def _test_ex8():
    # Solid red, green, blue patches → known luma values.
    red = t.zeros(2, 2, 3); red[..., 0] = 1.0
    green = t.zeros(2, 2, 3); green[..., 1] = 1.0
    blue = t.zeros(2, 2, 3); blue[..., 2] = 1.0
    assert t.allclose(ex8_rgb_to_grayscale(red),   t.full((2, 2), 0.299), atol=1e-5)
    assert t.allclose(ex8_rgb_to_grayscale(green), t.full((2, 2), 0.587), atol=1e-5)
    assert t.allclose(ex8_rgb_to_grayscale(blue),  t.full((2, 2), 0.114), atol=1e-5)
    # White → 1.0 (coefficients sum to 1).
    white = t.ones(3, 4, 3)
    g_white = ex8_rgb_to_grayscale(white)
    assert g_white.shape == (3, 4), f'expected (3,4), got {tuple(g_white.shape)}'
    assert t.allclose(g_white, t.ones(3, 4), atol=1e-5), 'white in → white out'
    # Black → 0.0.
    black = t.zeros(3, 4, 3)
    assert t.allclose(ex8_rgb_to_grayscale(black), t.zeros(3, 4), atol=1e-5), 'black in → black out'
    # Larger image just to validate it runs at scale.
    big = t.rand(64, 64, 3, generator=t.Generator().manual_seed(0))
    g_big = ex8_rgb_to_grayscale(big)
    assert g_big.shape == (64, 64)
    assert g_big.min().item() >= 0.0 and g_big.max().item() <= 1.0, 'luma must stay in [0, 1]'

    # --- Side-by-side visualization ---
    H, W = 64, 96
    ys = t.linspace(0, 1, H).unsqueeze(1).expand(H, W)
    xs = t.linspace(0, 1, W).unsqueeze(0).expand(H, W)
    synth = t.stack([
        xs,                  # R increases left→right
        ys,                  # G increases top→bottom
        1 - 0.5 * (xs + ys), # B falls off diagonally
    ], dim=-1).clamp(0, 1)
    synth_gray = ex8_rgb_to_grayscale(synth)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
    ax1.imshow(synth.numpy())
    ax1.set_title('original RGB')
    ax1.axis('off')
    ax2.imshow(synth_gray.numpy(), cmap='gray', vmin=0, vmax=1)
    ax2.set_title('luma grayscale (BT.601)')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()
    _dd_passed.add('ex8')
    print("ex8 ✓")

_test_ex8()

<details><summary>Solution</summary>

```python
def ex8_rgb_to_grayscale(img: Tensor) -> Tensor:
    r, g, b = img.unbind(dim=-1)
    return 0.299 * r + 0.587 * g + 0.114 * b
```

**Why luma weights are not equal.** The human eye is much more sensitive to green than to blue or red. The BT.601 coefficients (`0.299, 0.587, 0.114`) reflect that perceptual weighting — a naive `(r + g + b) / 3` produces a darker, washed-out grayscale.

**Why `unbind(dim=-1)` not `img[..., 0]`.** Both work, but the destructure reads like math: `r, g, b` are named, the order is explicit, and there's no chance of accidentally writing `[..., 1]` when you meant blue. For channel-last image tensors, `unbind(dim=-1)` is the idiomatic move.

**Sums to 1 → preserves brightness.** Because `0.299 + 0.587 + 0.114 == 1.0`, white maps to white and black maps to black. If you weight differently the output will be biased dim or bright.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()